# Customer Churn Prediction Notebook

Welcome! This notebook is the required starting point and submission format for the AI task.

## How this notebook works

- Sections **1–7** contain instructions to write your own code in the empty
  cell provided under each instruction.
- Section **8** is pre-written for evaluator use and **must not be modified**. We will run
  it against a **hidden test set** to score your submission.
- Your notebook must run **top-to-bottom without errors** before submission
  (`Restart Kernel & Run All`).

## 0. Setup

Standard imports, and file paths are set up below.
Add any additional imports your approach needs in the same cell.


In [31]:
import numpy as np
import pandas as pd
import tensorflow as tf


from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

# File paths
TRAIN_PATH = "dataset-1.csv"        # <-- set this to your local copy of the training CSV
TARGET_COLUMN = "churn"    # <-- set this to the churn label column name


EVALUATION_SPLIT_PATH = "./evaluation_split.csv"  # organizers only — do not rely on this file existing locally
df = pd.read_csv("dataset-1.csv")
print(df.head())

_state = {
    "columns":None,
    "num_col":None,
    "scaler" :None,
}


   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  4950-BDEUX    Male              0      No         No      35           No   
1  7993-NQLJE    Male              0     Yes        Yes      15          Yes   
2  7321-ZNSLA    Male              0     Yes        Yes      13           No   
3  4922-CVPDX  Female              0     Yes         No      26          Yes   
4  2903-YYTBW    Male              0     Yes        Yes       1          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...              Yes   
1                No     Fiber optic            Yes  ...               No   
2  No phone service             DSL            Yes  ...               No   
3                No             DSL             No  ...              Yes   
4                No             DSL             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape

## 1. Load the Data

Load the dataset from `TRAIN_PATH` into a dataframe.

In [32]:
df = pd.read_csv('/content/dataset-1.csv')

## 2. Explore and Clean the Dataset

Address data quality issues before modeling:

- Identify and handle missing / null values.
- Identify categorical variables and decide how each will be handled.
- Check for duplicate rows, inconsistent categories, and outliers.
- Document any rows/columns you drop and why.

In [33]:


def data_cleaning(df):
    df = df.copy()

    # Standardize column names
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
    print("Original shape:", df.shape)

    # Remove duplicate rows
    n_dupes = df.duplicated().sum()
    df = df.drop_duplicates()
    print(f"Dropped {n_dupes} duplicate rows (duplicates add no information)")

    # Drop ID-like columns (no predictive value)
    id_cols = [c for c in df.columns if c == "id" or c.endswith("_id") or c == "customerid"]
    df = df.drop(columns=id_cols, errors="ignore")
    print("Dropped ID columns:", id_cols)

    # Clean text values (strip spaces, lowercase) for consistent categories
    for c in df.select_dtypes("object").columns:
        df[c] = df[c].astype(str).str.strip().str.lower().replace({"nan": np.nan, "": np.nan})

    # Convert numeric-looking text columns (e.g. totalcharges) to numbers
    for c in df.select_dtypes("object").columns:
        if c == TARGET_COLUMN:
            continue
        conv = pd.to_numeric(df[c], errors="coerce")
        if conv.notna().mean() > 0.9:
            df[c] = conv

    # Convert target to 0/1
    if TARGET_COLUMN in df.columns and df[TARGET_COLUMN].dtype == "object":
        df[TARGET_COLUMN] = df[TARGET_COLUMN].map(
            {"yes": 1, "no": 0, "1": 1, "0": 0, "true": 1, "false": 0}
        )

    # Drop rows with missing target (can't train/evaluate without a label)
    if TARGET_COLUMN in df.columns:
        before = len(df)
        df = df.dropna(subset=[TARGET_COLUMN])
        print(f"Dropped {before - len(df)} rows with missing target")

    # Handle missing values
    feats = [c for c in df.columns if c != TARGET_COLUMN]
    num_cols = df[feats].select_dtypes("number").columns
    cat_cols = df[feats].select_dtypes(exclude="number").columns
    print("Missing values per column:\n", df.isnull().sum()[df.isnull().sum() > 0])

    for c in num_cols:
        df[c] = df[c].fillna(df[c].median())        # median is robust to outliers
    for c in cat_cols:
        mode = df[c].mode()
        df[c] = df[c].fillna(mode.iloc[0] if not mode.empty else "unknown")

    # Outliers: cap at 1st/99th percentile instead of dropping rows
    for c in num_cols:
        if df[c].nunique() > 2:                     # skip binary columns
            lo, hi = df[c].quantile([0.01, 0.99])
            df[c] = df[c].clip(lo, hi)

    print("Cleaned shape:", df.shape)
    print("Categorical columns:", list(cat_cols))
    return df



## 3. Feature Engineering

Transform the raw columns into features that are more useful for a classification model.

For example: combining related columns, extracting information from
dates or creating new columns from existing ones.

In [34]:
def feature_engineering(df):
  df = df.copy()
  if {"totalcharges", "tenure"} <= set(df.columns):
    df["avg_monthly_spend"] = df["totalcharges"] / (df["tenure"] + 1)
  if {"monthlycharges", "tenure"} <= set(df.columns):
    df["charge_x_tenure"] = df["monthlycharges"] * df["tenure"]
  if "tenure" in df.columns:
    df["tenure_bucket"] = pd.cut(df["tenure"], [-1, 12, 24, 48, 1e9],
  labels=["0-1y", "1-2y", "2-4y", "4y+"]).astype(str)
  return df


## 4. Transform, Normalize, and Scale Features

Prepare the final feature set for modeling:

- Encode categorical variables.
- Scale/normalize numeric features as appropriate for the model(s) you plan to use.

In [35]:
def scale_features(df):
  df = df.copy()
  y = df.pop(TARGET_COLUMN) if TARGET_COLUMN in df.columns else None

  df = pd.get_dummies(df, drop_first=True, dtype=float)

  if _state["columns"] is None: # first call = training data
    _state["columns"] = df.columns.tolist()
    _state["num_cols"] = [c for c in df.columns if df[c].nunique() > 2]
    _state["scaler"] = StandardScaler().fit(df[_state["num_cols"]])
  else: # later calls: align to training columns
    df = df.reindex(columns=_state["columns"], fill_value=0.0)

    df[_state["num_cols"]] = _state["scaler"].transform(df[_state["num_cols"]])
  if y is not None:
    df[TARGET_COLUMN] = y.values
    return df



In [36]:
# Do not modify this function
def preprocess_data(df):
    df = df.copy()
    df = data_cleaning(df)
    df = feature_engineering(df)
    df = scale_features(df)

    return df

## 5. Train / Validation / Test Split

Split your cleaned data into three sets:

- **Train** — used to fit models.
- **Validation** — used for hyperparameter tuning and model comparison.
- **Test** — held out until Section 6.

In [37]:
def split_data(df):
    train_split, temp_split = train_test_split(
        df,
        test_size=0.3,
        random_state=42,
        stratify=df[TARGET_COLUMN],
    )
    validation_split, test_split = train_test_split(
        temp_split,
        test_size=0.5,
        random_state=42,
        stratify=temp_split[TARGET_COLUMN],
    )
    return train_split, validation_split, test_split

## 6. Model Training, and Hyperparameter Tuning

- Train classification models on the train set (try more than one algorithm and compare).
- Tune hyperparameters using the validation set.
- Use the `evaluate_predictions()` function (defined in Section 0) to report **train** and **validation**
  metrics for each model: Accuracy, Precision, Recall, F1-score, ROC-AUC.
- Select your best-performing model/configuration.

### Required output of this section
- **`predict_churn(df)`** — takes a raw dataframe, calls `preprocess_data(df)` internally, then returns `(predictions, probabilities)` from your best trained model.

This is what Section 8 will call on the hidden test set, so both functions must work on data that hasn't been touched yet — don't rely on variables that only exist because you already cleaned `df` earlier in the notebook.

In [38]:
def predict_churn(df):
    # 1. Preprocess and split training data
    df_raw = pd.read_csv(TRAIN_PATH)
    df_clean = preprocess_data(df_raw)

    train_split, val_split, test_split = split_data(df_clean)

    X_train = train_split.drop(columns=[TARGET_COLUMN])
    y_train = train_split[TARGET_COLUMN]

    # 2. Train your best-performing model
    from sklearn.ensemble import RandomForestClassifier

    model = RandomForestClassifier(random_state=42)
    model.fit(X_train, y_train)

    # 3. Define predict_churn properly
    def predict_churn(df):
        # Preprocess incoming raw test dataframe
        processed = preprocess_data(df)


        if TARGET_COLUMN in processed.columns:
            processed = processed.drop(columns=[TARGET_COLUMN])


## 7. Evaluate on the Held-Out Test Set

Run `predict_churn()` on your held-out test split (never used in training or tuning) and report
the same metrics using `evaluate_predictions()`.

In [39]:
def evaluate_predictions(y_true, y_pred, y_proba):
    # DO NOT modify
    # Standard metric report used throughout this notebook.
    # Both your own evaluation (Section 6) and the grading step (Section 8) use this function.
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
    }
    print("--- Metrics ---")
    for name, value in metrics.items():
        print(f"{name:>10}: {value:.4f}")
    return metrics

---
## 8. Standardized Evaluation — DO NOT MODIFY

Everything below is run by the organizers for every submission. It loads the hidden evaluation set
and scores your submission. Make sure your notebook runs top-to-bottom.


In [40]:
# =====================================================================
# ORGANIZER EVALUATION CELL — DO NOT MODIFY
# =====================================================================
assert "predict_churn" in dir(), (
    "predict_churn(df) is not defined. Your submission must define this "
    "function in Section 6 (see instructions there)."
)

evaluation_df = pd.read_csv(EVALUATION_SPLIT_PATH)
X_hidden = evaluation_df.drop(columns=[TARGET_COLUMN])
y_hidden = evaluation_df[TARGET_COLUMN]

y_pred, y_proba = predict_churn(X_hidden)

evaluation_metrics = evaluate_predictions(y_hidden, y_pred, y_proba)
pd.DataFrame([evaluation_metrics])

FileNotFoundError: [Errno 2] No such file or directory: './evaluation_split.csv'